In [25]:
# Import libraries
import panel as pn
import pandas as pd
import geopandas as gpd
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path
    

In [38]:
import pandas as pd

# Load the CSV with the second row as headers
df = pd.read_csv('/workspaces/primary-education/primary_education_dataset/primary_indicators.csv', header=1)

# Rename columns with year suffixes
columns = df.columns
new_columns = [
    'District',
    'Gross Enrollment Ratio_2014', 'Pupil Teacher Ratio_2014', 'Primary Completion Ratio_2014',
    'Gross Enrollment Ratio_2015', 'Pupil Teacher Ratio_2015', 'Primary Completion Ratio_2015',
    'Gross Enrollment Ratio_2016', 'Pupil Teacher Ratio_2016', 'Primary Completion Ratio_2016',
    'Gross Enrollment Ratio_2017', 'Pupil Teacher Ratio_2017', 'Primary Completion Ratio_2017'
]
df.columns = new_columns

# Convert to long format
df_long = pd.melt(
    df,
    id_vars=['District'],
    value_vars=[
        'Gross Enrollment Ratio_2014', 'Pupil Teacher Ratio_2014', 'Primary Completion Ratio_2014',
        'Gross Enrollment Ratio_2015', 'Pupil Teacher Ratio_2015', 'Primary Completion Ratio_2015',
        'Gross Enrollment Ratio_2016', 'Pupil Teacher Ratio_2016', 'Primary Completion Ratio_2016',
        'Gross Enrollment Ratio_2017', 'Pupil Teacher Ratio_2017', 'Primary Completion Ratio_2017'
    ],
    var_name='Metric_Year',
    value_name='Value'
)

# Split Metric_Year into Metric and Year
df_long[['Metric', 'Year']] = df_long['Metric_Year'].str.extract(r'(.*)_(\d{4})')
df_long = df_long.drop(columns=['Metric_Year'])

# Clean and standardize
df_long['Metric'] = df_long['Metric'].str.strip()
df_long['Value'] = pd.to_numeric(df_long['Value'], errors='coerce')
df_long['District'] = df_long['District'].str.upper()

# Filter to match GeoJSON districts
with open('uganda_districts.json') as f:
    uganda_geojson = json.load(f)
geo_districts = [f['properties']['District'] for f in uganda_geojson['features']]
df_long = df_long[df_long['District'].isin(geo_districts)]

# Save the full dataset
df_long.to_csv('education_data_long.csv', index=False)

# Verify
print(df_long.head())
print(df_long['District'].unique())

       District  Value                  Metric  Year
0        BUIKWE   85.0  Gross Enrollment Ratio  2014
1  BUKOMANSIMBI  131.0  Gross Enrollment Ratio  2014
2     BUTAMBALA  125.0  Gross Enrollment Ratio  2014
3        BUVUMA  198.0  Gross Enrollment Ratio  2014
4         GOMBA  111.0  Gross Enrollment Ratio  2014
['BUIKWE' 'BUKOMANSIMBI' 'BUTAMBALA' 'BUVUMA' 'GOMBA' 'KALANGALA'
 'KALUNGU' 'KAMPALA' 'KAYUNGA' 'KIBOGA' 'KYOTERA' 'LWENGO' 'LYANTONDE'
 'MASAKA' 'MITYANA' 'MPIGI' 'MUBENDE' 'MUKONO' 'NAKASEKE' 'NAKASONGOLA'
 'RAKAI' 'SSEMBABULE' 'WAKISO' 'BUDAKA' 'BUDUDA' 'BUGIRI' 'BUKEDEA'
 'BUKWO' 'BULAMBULI' 'BUSIA' 'BUTALEJA' 'BUYENDE' 'IGANGA' 'JINJA'
 'KABERAMAIDO' 'KALIRO' 'KAMULI' 'KAPCHORWA' 'KATAKWI' 'KIBUKU' 'KUMI'
 'KWEEN' 'LUUKA' 'MANAFWA' 'MAYUGE' 'MBALE' 'NAMAYINGO' 'NAMISINDWA'
 'NAMUTUMBA' 'NGORA' 'PALLISA' 'SERERE' 'SIRONKO' 'SOROTI' 'TORORO' 'ABIM'
 'ADJUMANI' 'AGAGO' 'ALEBTONG' 'AMOLATAR' 'AMUDAT' 'AMURIA' 'AMURU' 'APAC'
 'ARUA' 'DOKOLO' 'GULU' 'KAABONG' 'KITGUM' 'KOBO

In [40]:
df_long.to_csv('education_data_long.csv', index=False)

# Verify the saved data
df_verified = pd.read_csv('education_data_long.csv')
print(df_verified.head())
print(df_verified['District'].unique())
print(f"Total rows: {len(df_verified)}")
print(f"Unique districts: {len(df_verified['District'].unique())}")
print(f"Unique metrics: {df_verified['Metric'].unique()}")
print(f"Unique years: {df_verified['Year'].unique()}")

       District  Value                  Metric  Year
0        BUIKWE   85.0  Gross Enrollment Ratio  2014
1  BUKOMANSIMBI  131.0  Gross Enrollment Ratio  2014
2     BUTAMBALA  125.0  Gross Enrollment Ratio  2014
3        BUVUMA  198.0  Gross Enrollment Ratio  2014
4         GOMBA  111.0  Gross Enrollment Ratio  2014
['BUIKWE' 'BUKOMANSIMBI' 'BUTAMBALA' 'BUVUMA' 'GOMBA' 'KALANGALA'
 'KALUNGU' 'KAMPALA' 'KAYUNGA' 'KIBOGA' 'KYOTERA' 'LWENGO' 'LYANTONDE'
 'MASAKA' 'MITYANA' 'MPIGI' 'MUBENDE' 'MUKONO' 'NAKASEKE' 'NAKASONGOLA'
 'RAKAI' 'SSEMBABULE' 'WAKISO' 'BUDAKA' 'BUDUDA' 'BUGIRI' 'BUKEDEA'
 'BUKWO' 'BULAMBULI' 'BUSIA' 'BUTALEJA' 'BUYENDE' 'IGANGA' 'JINJA'
 'KABERAMAIDO' 'KALIRO' 'KAMULI' 'KAPCHORWA' 'KATAKWI' 'KIBUKU' 'KUMI'
 'KWEEN' 'LUUKA' 'MANAFWA' 'MAYUGE' 'MBALE' 'NAMAYINGO' 'NAMISINDWA'
 'NAMUTUMBA' 'NGORA' 'PALLISA' 'SERERE' 'SIRONKO' 'SOROTI' 'TORORO' 'ABIM'
 'ADJUMANI' 'AGAGO' 'ALEBTONG' 'AMOLATAR' 'AMUDAT' 'AMURIA' 'AMURU' 'APAC'
 'ARUA' 'DOKOLO' 'GULU' 'KAABONG' 'KITGUM' 'KOBO